# sat_beyond_health — Satélite Beyond Health

Este notebook construye el **satélite de Beyond Health**, que es la tabla central donde
se consolida toda la información de los asegurados del sistema Beyond Health.

## ¿Qué hace este notebook?

Toma información de **6 tablas fuente** del sistema Beyond Health y las une en una sola tabla
que contiene los datos completos de cada asegurado, identificando claramente si es
**titular** (el asegurado principal del contrato) o **beneficiario** (familiar u otra
persona incluida en el contrato).

## Tablas fuente

| Tabla | Qué contiene |
|---|---|
| `bh_sa_member` | Lista de todos los miembros (titulares y beneficiarios) de cada contrato |
| `bh_sa_affiliation_contract` | Información del contrato de afiliación |
| `bh_sa_person` | Datos personales de cada persona (nombre, documento, etc.) |
| `bh_sa_institution` | Datos de la institución o empresa que suscribe el contrato |
| `bh_sa_address` | Direcciones registradas de cada persona |
| `bh_sa_city` | Catálogo de ciudades con su código y nombre |

## Tabla destino

`uc_axa_cli.silver.sat_beyond_health`

## Regla de negocio clave

- **Titular**: la persona se obtiene desde el **contrato** (`aco.per_ncode`)
- **Beneficiario**: la persona se obtiene desde el **member** (`mem.per_ncode`)

## Paso 1 — Preparar la dirección residencial de cada persona

Antes de armar el satélite, necesitamos tener lista la **dirección residencial** de cada
persona. El problema es que una persona puede tener varias direcciones registradas
(residencial, laboral, de correo, etc.), por eso hacemos este paso previo.

**¿Qué hacemos aquí?**

1. De la tabla de direcciones (`bh_sa_address`), filtramos solo las que son de tipo
   **residencial** (`lty_ncode = 1`).
2. Cruzamos con el catálogo de ciudades para traer el **código de ciudad**.
3. Agrupamos por persona para quedarnos con **una sola fila por persona**
   (usamos `MAX` para tomar un valor cuando hay más de uno).

El resultado es una tabla auxiliar `residencial` con una fila por persona que tiene:
su dirección residencial y el código de su ciudad.

## Paso 2 — Preparar el nombre de la ciudad

Con el código de ciudad que obtuvimos en el paso anterior, buscamos el
**nombre legible de la ciudad** en el catálogo `bh_sa_city`.

Esto nos permite mostrar, por ejemplo, `"BOGOTA D.C."` en lugar de solo el código `"11001"`.

## Paso 3 — Construir los registros de TITULARES

Aquí construimos la información completa de los **titulares** del contrato.

**¿Cómo identificamos al titular?**  
En la tabla `bh_sa_member`, el titular es el miembro cuyo `per_ncode` coincide con el
`per_ncode` del contrato de afiliación (`mem.per_ncode = aco.per_ncode`). Esto garantiza
**exactamente un registro TITULAR por contrato**, sin importar cuántos beneficiarios tenga.

Los datos personales (nombre, documento, dirección) se obtienen desde el **contrato**
(`aco.per_ncode`), porque el titular es quien firma el contrato.

**Joins que se hacen:**

| Unión | Para qué |
|---|---|
| `member` + `contrato` | Traer los datos del contrato al que pertenece cada miembro |
| `contrato` + `persona` | Traer los datos personales del **titular del contrato** |
| `contrato` + `institución` | Traer los datos de la empresa u organización del contrato |
| `contrato` + `residencial` | Traer la dirección del **titular** (via `aco.per_ncode`) |
| `residencial` + `ciudad` | Traer el nombre de la ciudad del titular |

**Filtro clave:** `WHERE mem.per_ncode = aco.per_ncode`

Al final, cada fila queda marcada con `rol = 'TITULAR'`.

**Nota sobre los nombres de columnas:**  
Cuando una misma columna existe en varias tablas (por ejemplo `fec_cargue` existe en
member, contrato y persona), se le agrega un prefijo para saber de dónde viene:
`mem_fec_cargue`, `aco_fec_cargue`, `per_fec_cargue`.

## Paso 4 — Construir los registros de BENEFICIARIOS

Aquí construimos la información completa de los **beneficiarios** del contrato.

**¿En qué se diferencia del titular?**  
Hay **tres diferencias** respecto al bloque anterior:

1. **Filtro de rol:** Solo se incluyen miembros cuyo `per_ncode` es **distinto** al del
   contrato (`WHERE mem.per_ncode <> aco.per_ncode`). Contratos sin beneficiarios no
   producen ninguna fila en este CTE.
2. La persona del beneficiario viene del **member** (`mem.per_ncode`), no del contrato.
3. La dirección residencial también se busca con el `per_ncode` del **member**.

Un contrato puede tener **0, 1 o N beneficiarios** — cada beneficiario genera su propia
fila con su documento y datos personales únicos.

Todo lo demás (columnas, estructura, prefijos) es idéntico al bloque de titulares.  
Al final, cada fila queda marcada con `rol = 'BENEFICIARIO'`.

## Paso 5 — Unir titulares y beneficiarios en una sola tabla

Con `UNION ALL` juntamos los registros de titulares y beneficiarios en una sola tabla.

La columna `rol` nos permite distinguir en todo momento quién es quién:
- `TITULAR` → persona principal del contrato (un registro por contrato)
- `BENEFICIARIO` → familiar u otro miembro incluido en el contrato (0 a N registros por contrato)

**Resultado esperado:** El total de filas es igual al total de miembros en `bh_sa_member`,
ya que los filtros `mem.per_ncode = aco.per_ncode` y `mem.per_ncode <> aco.per_ncode`
son mutuamente excluyentes y exhaustivos — cada fila de member cae exactamente en uno de los dos grupos.

## Ejecución — Crear el satélite Beyond Health

La siguiente celda ejecuta todo el proceso descrito arriba en un solo paso.
Crea (o reemplaza) la tabla `uc_axa_cli.silver.sat_beyond_health` con todos los datos.

> ⚠️ **Importante:** Este proceso borra y recrea la tabla completa cada vez que se ejecuta.
> Esto garantiza que siempre refleje el estado más reciente de las tablas fuente.

In [ ]:
%sql

CREATE OR REPLACE TABLE axa_col_slv_dv.stg_cliente.sat_beyond_health
USING DELTA AS

-- ============================================================
-- PASO 1: Dirección residencial por persona
-- ============================================================
WITH residencial AS (
    SELECT
        a.per_ncode             AS res_per_ncode,
        MAX(a.add_caddress)     AS dir_res,
        MAX(c.cit_clegalcode)   AS ciu_res_codigo
    FROM axa_col_slv_dv.core_bh.bh_sa_address a
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_city c
        ON a.cit_ncode = c.cit_ncode
    WHERE a.lty_ncode = 1
    GROUP BY a.per_ncode
),

-- ============================================================
-- PASO 2: Ciudad y país
-- bh_sa_country solo tiene 2 registros: Colombia y Exterior.
-- dep_ncode presente en bh_sa_city → ciudad colombiana → Colombia.
-- dep_ncode ausente → ciudad extranjera → Exterior.
-- pais queda NULL cuando no hay ciudad registrada (LEFT JOIN).
-- ============================================================
ciudad AS (
    SELECT
        c.cit_clegalcode        AS ciu_res_codigo,
        c.cit_cname             AS ciudad_residencia,
        c.dep_ncode             AS dep_ncode,
        CASE
            WHEN c.dep_ncode IS NOT NULL THEN 'Colombia'
            ELSE 'Exterior'
        END                     AS pais
    FROM axa_col_slv_dv.core_bh.bh_sa_city c
),

-- ============================================================
-- PASO 3: Titulares (mem.per_ncode = aco.per_ncode)
-- ============================================================
titular AS (
    SELECT
        -- Llaves y relaciones
        mem.mem_ncode                           AS mem_ncode,
        mem.per_ncode                           AS mem_per_ncode,
        mem.aco_ncode                           AS mem_aco_ncode,
        aco.aco_ncode                           AS aco_ncode,
        aco.per_ncode                           AS aco_per_ncode,
        aco.ins_ncode                           AS aco_ins_ncode,
        per.per_ncode                           AS per_ncode,
        ins.ins_ncode                           AS ins_ncode,
        -- Datos personales
        per.TID_NCODE                           AS tipo_documento,
        per.PER_CIDENTIFICATIONNUMBER           AS numero_documento,
        per.PER_CFIRSTNAME                      AS primer_nombre,
        per.PER_CLASTNAME                       AS primer_apellido,
        per.PER_CMIDDLENAME                     AS segundo_nombre,
        per.PER_CMOTHERNAME                     AS segundo_apellido,
        COALESCE(per.PER_CEMAIL, per.PER_CMAIL) AS correo,
        per.PER_CMOBILEPHONE                    AS celular,
        per.PER_DBIRTHDATE                      AS fecha_nacimiento,
        per.PER_CGENDER                         AS genero,
        per.PER_BAUTHPERSONALINFO               AS ATDP,
        per.EAC_NCODE                           AS actividad_economica,
        per.MST_NCODE                           AS estado_civil,
        per.FECHA_CARGUE                        AS per_fecha_cargue,
        -- Institución
        ins.INS_CNAME                           AS nombre_completo_razon_social,
        ins.INS_CLEGALCODE                      AS eps,
        ins.INS_BEMAIL_SEND                     AS preferencia_contacto_email,
        ins.INS_BSMS_SEND                       AS preferencia_contacto_sms,
        -- Dirección residencial
        res.dir_res                             AS direccion_residencial,
        res.ciu_res_codigo                      AS codigo_ciudad,
        ciudad.ciudad_residencia,
        ciudad.dep_ncode,
        ciudad.pais,
        -- Contrato
        aco.pla_ncode                           AS plan,
        -- Rol
        'TITULAR'                               AS rol
    FROM axa_col_slv_dv.core_bh.bh_sa_member mem
    INNER JOIN axa_col_slv_dv.core_bh.bh_sa_affiliation_contract aco
        ON mem.aco_ncode = aco.aco_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_person per
        ON aco.per_ncode = per.per_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_institution ins
        ON aco.ins_ncode = ins.ins_ncode
    LEFT JOIN residencial res
        ON aco.per_ncode = res.res_per_ncode
    LEFT JOIN ciudad
        ON res.ciu_res_codigo = ciudad.ciu_res_codigo
    WHERE mem.per_ncode = aco.per_ncode
      AND per.FECHA_CARGUE >= ADD_MONTHS(CURRENT_DATE(), -6)
),

-- ============================================================
-- PASO 4: Beneficiarios (mem.per_ncode <> aco.per_ncode)
-- ============================================================
beneficiario AS (
    SELECT
        -- Llaves y relaciones
        mem.mem_ncode                           AS mem_ncode,
        mem.per_ncode                           AS mem_per_ncode,
        mem.aco_ncode                           AS mem_aco_ncode,
        aco.aco_ncode                           AS aco_ncode,
        aco.per_ncode                           AS aco_per_ncode,
        aco.ins_ncode                           AS aco_ins_ncode,
        per.per_ncode                           AS per_ncode,
        ins.ins_ncode                           AS ins_ncode,
        -- Datos personales
        per.TID_NCODE                           AS tipo_documento,
        per.PER_CIDENTIFICATIONNUMBER           AS numero_documento,
        per.PER_CFIRSTNAME                      AS primer_nombre,
        per.PER_CLASTNAME                       AS primer_apellido,
        per.PER_CMIDDLENAME                     AS segundo_nombre,
        per.PER_CMOTHERNAME                     AS segundo_apellido,
        COALESCE(per.PER_CEMAIL, per.PER_CMAIL) AS correo,
        per.PER_CMOBILEPHONE                    AS celular,
        per.PER_DBIRTHDATE                      AS fecha_nacimiento,
        per.PER_CGENDER                         AS genero,
        per.PER_BAUTHPERSONALINFO               AS ATDP,
        per.EAC_NCODE                           AS actividad_economica,
        per.MST_NCODE                           AS estado_civil,
        per.FECHA_CARGUE                        AS per_fecha_cargue,
        -- Institución
        ins.INS_CNAME                           AS nombre_completo_razon_social,
        ins.INS_CLEGALCODE                      AS eps,
        ins.INS_BEMAIL_SEND                     AS preferencia_contacto_email,
        ins.INS_BSMS_SEND                       AS preferencia_contacto_sms,
        -- Dirección residencial
        res.dir_res                             AS direccion_residencial,
        res.ciu_res_codigo                      AS codigo_ciudad,
        ciudad.ciudad_residencia,
        ciudad.dep_ncode,
        ciudad.pais,
        -- Contrato
        aco.pla_ncode                           AS plan,
        -- Rol
        'BENEFICIARIO'                          AS rol
    FROM axa_col_slv_dv.core_bh.bh_sa_member mem
    INNER JOIN axa_col_slv_dv.core_bh.bh_sa_affiliation_contract aco
        ON mem.aco_ncode = aco.aco_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_person per
        ON mem.per_ncode = per.per_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_institution ins
        ON aco.ins_ncode = ins.ins_ncode
    LEFT JOIN residencial res
        ON mem.per_ncode = res.res_per_ncode
    LEFT JOIN ciudad
        ON res.ciu_res_codigo = ciudad.ciu_res_codigo
    WHERE mem.per_ncode <> aco.per_ncode
      AND per.FECHA_CARGUE >= ADD_MONTHS(CURRENT_DATE(), -6)
)

-- ============================================================
-- PASO 5: Resultado final — titulares + beneficiarios
-- ============================================================
SELECT * FROM titular
UNION ALL
SELECT * FROM beneficiario

## Validación — Verificar el resultado

Ejecuta las siguientes celdas para confirmar que el satélite quedó cargado correctamente.

In [ ]:
%sql
-- Conteo de filas por rol
SELECT
    rol,
    COUNT(*) AS total_filas
FROM axa_col_slv_dv.stg_cliente.sat_beyond_health
GROUP BY rol
ORDER BY rol

In [ ]:
%sql
-- Total de filas del satélite
SELECT COUNT(*) AS total_filas
FROM axa_col_slv_dv.stg_cliente.sat_beyond_health

In [ ]:
%sql
-- Vista previa de los primeros 5 registros
SELECT
    mem_ncode,
    tipo_documento,
    numero_documento,
    rol,
    dir_res,
    ciudad_residencia,
    pais
FROM axa_col_slv_dv.stg_cliente.sat_beyond_health
LIMIT 5

In [ ]:
-- RQ-001: Celular de beneficiario menor de edad
-- Muestra los registros donde el celular del beneficiario
-- debe ser reemplazado por el celular del titular de su póliza.

SELECT
    ben.mem_ncode                                               AS mem_ncode_beneficiario,
    ben.aco_ncode                                               AS aco_ncode,
    ben.numero_documento                                        AS doc_beneficiario,
    ben.primer_nombre || ' ' || ben.primer_apellido             AS nombre_beneficiario,
    FLOOR(MONTHS_BETWEEN(CURRENT_DATE(), ben.fecha_nacimiento) / 12) AS edad,
    ben.celular                                                 AS celular_actual,
    tit.celular                                                 AS celular_correcto,
    tit.numero_documento                                        AS doc_titular,
    tit.primer_nombre || ' ' || tit.primer_apellido             AS nombre_titular
FROM ${rq001.tabla} ben
INNER JOIN ${rq001.tabla} tit
    ON  ben.aco_ncode = tit.aco_ncode
    AND tit.rol       = 'TITULAR'
WHERE ben.rol = 'BENEFICIARIO'
  AND FLOOR(MONTHS_BETWEEN(CURRENT_DATE(), ben.fecha_nacimiento) / 12) < ${rq001.edad_minima}
ORDER BY ben.aco_ncode, ben.edad

In [ ]:
# ============================================================
# RQ-001 — Parámetros
# ============================================================
tabla      = "axa_col_slv_dv.stg_cliente.sat_beyond_health"
edad_minima = 18

spark.sql(f"SET rq001.tabla       = {tabla}")
spark.sql(f"SET rq001.edad_minima = {edad_minima}")

### Regla RQ-001 — Celular de beneficiario menor de edad

**Descripción:** Si el beneficiario tiene menos de `edad_minima` años, el campo `celular`
debe tener el número del **titular de su póliza**, no el suyo propio.

**Parámetros configurables:**
| Parámetro | Descripción | Valor por defecto |
|---|---|---|
| `tabla` | Tabla del satélite | `axa_col_slv_dv.stg_cliente.sat_beyond_health` |
| `edad_minima` | Edad mínima para usar celular propio | `18` |

**Resultado:** Filas de beneficiarios menores de edad con el celular actual y el celular correcto (del titular).

---

## Reglas de Calidad

Las siguientes celdas son **queries independientes y parametrizables** para evaluar y corregir reglas de calidad sobre el satélite.

Cada regla:
- Tiene sus **parámetros** en una celda Python separada (fácil de ajustar sin tocar el SQL)
- Produce un resultado con las columnas afectadas y el valor correcto sugerido
- Se puede ejecutar de forma independiente del proceso de creación del satélite